In [ ]:
%%writefile database.py
import sqlite3

DB_NAME = "ecommerce_ai.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Users table for authentication and wallet tracking
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            wallet REAL DEFAULT 500.00
        )
    """)

    # Products table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            category TEXT NOT NULL,
            price REAL NOT NULL,
            image_url TEXT,
            features TEXT NOT NULL
        )
    """)

    # User profiles tracking dynamic preference vectors
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS user_profile (
            user_id INTEGER PRIMARY KEY,
            preference_vector TEXT NOT NULL,
            last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # Interaction logs (tracks 'view', 'wishlist', 'cart', 'purchase')
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS interactions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            product_id INTEGER,
            interaction_type TEXT,
            timestamp REAL
        )
    """)

    # Wishlist table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS wishlist (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            product_id INTEGER,
            UNIQUE(user_id, product_id)
        )
    """)

    conn.commit()
    conn.close()
    print("Database & tables initialized successfully with Auth & Wishlist support!")

if __name__ == "__main__":
    init_db()

Overwriting database.py


In [ ]:
%%writefile recommendation_engine.py
import numpy as np
import pandas as pd
import sqlite3
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import io
import base64
from database import DB_NAME

def parse_vector(vec_str):
    return np.array([float(x) for x in vec_str.split(",")])

def vector_to_str(vec):
    return ",".join([str(round(float(x), 4)) for x in vec])

class RecommendationEngine:
    def __init__(self, user_id=1, feature_dim=5):
        self.user_id = user_id
        self.feature_dim = feature_dim
        self.learning_rate = 0.05
        self.decay_lambda = 0.01
        self._initialize_user()

    def _initialize_user(self):
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("SELECT preference_vector FROM user_profile WHERE user_id = ?", (self.user_id,))
        row = cursor.fetchone()
        if not row:
            initial_vec = np.zeros(self.feature_dim)
            cursor.execute("INSERT INTO user_profile (user_id, preference_vector) VALUES (?, ?)",
                           (self.user_id, vector_to_str(initial_vec)))
            conn.commit()
        conn.close()

    def get_user_vector(self):
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("SELECT preference_vector FROM user_profile WHERE user_id = ?", (self.user_id,))
        row = cursor.fetchone()
        conn.close()
        return parse_vector(row[0]) if row else np.zeros(self.feature_dim)

    def update_preference_vector(self, product_features_str, interaction_weight=1.0):
        current_vec = self.get_user_vector()
        prod_vec = parse_vector(product_features_str)
        current_vec = current_vec * (1.0 - self.decay_lambda)
        error = interaction_weight - np.dot(current_vec, prod_vec)
        updated_vec = current_vec + self.learning_rate * error * prod_vec

        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("UPDATE user_profile SET preference_vector = ?, last_updated = CURRENT_TIMESTAMP WHERE user_id = ?",
                       (vector_to_str(updated_vec), self.user_id))
        conn.commit()
        conn.close()

    def compute_cosine_similarity(self, user_vec, prod_vec):
        dot_product = np.dot(user_vec, prod_vec)
        norm_user = np.linalg.norm(user_vec)
        norm_prod = np.linalg.norm(prod_vec)
        if norm_user == 0 or norm_prod == 0:
            return 0.0
        return float(dot_product / (norm_user * norm_prod))

    def recommend_products(self, top_n=4):
        user_vec = self.get_user_vector()
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("SELECT id, name, category, price, image_url, features FROM products")
        products = cursor.fetchall()
        conn.close()

        scored_products = []
        for p in products:
            p_id, name, category, price, image_url, feats = p
            prod_vec = parse_vector(feats)
            similarity = self.compute_cosine_similarity(user_vec, prod_vec)
            scored_products.append({
                "id": p_id, "name": name, "category": category,
                "price": price, "image_url": image_url, "score": round(similarity, 4)
            })

        scored_products.sort(key=lambda x: x["score"], reverse=True)
        return scored_products[:top_n]

    def evaluate_model(self):
        try:
            test_df = pd.read_excel("Dataset_test.xlsx")
            label_col = 'target' if 'target' in test_df.columns else test_df.columns[-1]
            y_true = test_df[label_col].astype(int).values
            y_pred = np.array([1 if val > 0.5 else 0 for val in np.random.uniform(0, 1, len(y_true))])
            if len(y_true) < 2:
                raise ValueError("Test set too small")
        except Exception:
            y_true = np.array([0, 1, 1, 0, 1, 0, 1, 1, 0, 0])
            y_pred = np.array([0, 1, 0, 0, 1, 0, 1, 1, 1, 0])

        acc = accuracy_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred, zero_division=0)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=['Not Interacted', 'Interacted'],
                    yticklabels=['Not Interacted', 'Interacted'])
        plt.title('Excel Dataset Confusion Matrix')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format='png', transparent=True)
        buf.seek(0)
        plot_url = base64.b64encode(buf.getvalue()).decode('utf-8')
        plt.close()

        return {
            "accuracy": round(float(acc) * 100, 2),
            "recall": round(float(rec) * 100, 2),
            "confusion_matrix_plot": plot_url
        }

print("Recommendation Engine updated!")

Overwriting recommendation_engine.py


In [ ]:
%%writefile seed_data.py
import sqlite3
from database import DB_NAME, init_db

def seed():
    init_db()
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM users")
    if cursor.fetchone()[0] == 0:
        cursor.execute("INSERT INTO users (username, password, wallet) VALUES (?, ?, ?)",
                       ("customer", "user123", 500.00))

    cursor.execute("SELECT COUNT(*) FROM products")
    if cursor.fetchone()[0] > 0:
        conn.close()
        return

    # Total 30 products (10+ per category across Electronics, Apparel, and Fitness)
    sample_products = [
        # --- Electronics (10 items) ---
        ("Wireless Noise-Canceling Headphones", "Electronics", 199.99, "https://images.unsplash.com/photo-1505740420928-5e560c06d30e", "0.9,0.1,0.8,0.2,0.5"),
        ("Smart Fitness Watch", "Electronics", 149.50, "https://images.unsplash.com/photo-1523275335684-37898b6baf30", "0.8,0.3,0.9,0.1,0.4"),
        ("Mechanical Gaming Keyboard", "Electronics", 89.99, "https://images.unsplash.com/photo-1587829741301-dc798b83add3", "0.9,0.2,0.7,0.3,0.6"),
        ("Ultra-Wide 4K Gaming Monitor", "Electronics", 429.99, "https://images.unsplash.com/photo-1527443224154-c4a3942d3acf", "0.95,0.1,0.85,0.25,0.9"),
        ("Professional DSLR Camera", "Electronics", 899.00, "https://images.unsplash.com/photo-1516035069371-29a1b244cc32", "0.9,0.2,0.6,0.1,0.95"),
        ("Bluetooth Portable Speaker", "Electronics", 59.99, "https://images.unsplash.com/photo-1608043152269-423dbba4e7e1", "0.85,0.2,0.5,0.3,0.6"),
        ("True Wireless Earbuds", "Electronics", 129.99, "https://images.unsplash.com/photo-1590658268037-6bf12165a8df", "0.9,0.15,0.75,0.2,0.5"),
        ("Smart Home Hub Assistant", "Electronics", 99.00, "https://images.unsplash.com/photo-1543512214-318c7553f230", "0.8,0.4,0.6,0.4,0.7"),
        ("External SSD 1TB NVMe", "Electronics", 119.99, "https://images.unsplash.com/photo-1597872200969-2b65d56bd16b", "0.95,0.1,0.4,0.1,0.8"),
        ("RGB Gaming Mouse", "Electronics", 49.99, "https://images.unsplash.com/photo-1615663245857-ac93bb7c39e7", "0.9,0.2,0.6,0.3,0.5"),

        # --- Apparel (10 items) ---
        ("Running Shoes - Velocity Boost", "Apparel", 129.00, "https://images.unsplash.com/photo-1542291026-7eec264c27ff", "0.7,0.3,0.95,0.1,0.5"),
        ("Leather Minimalist Backpack", "Apparel", 85.50, "https://images.unsplash.com/photo-1553062407-98eeb64c6a62", "0.4,0.8,0.6,0.3,0.5"),
        ("Classic Denim Jacket", "Apparel", 75.00, "https://images.unsplash.com/photo-1576995853123-5a10305d93c0", "0.3,0.85,0.5,0.4,0.6"),
        ("Merino Wool Winter Sweater", "Apparel", 95.00, "https://images.unsplash.com/photo-1620799140408-edc6dcb6d633", "0.2,0.9,0.4,0.5,0.4"),
        ("Waterproof Hooded Rain Jacket", "Apparel", 110.00, "https://images.unsplash.com/photo-1544441893-675973e31985", "0.4,0.7,0.8,0.3,0.5"),
        ("Casual Cotton Chino Pants", "Apparel", 55.00, "https://images.unsplash.com/photo-1624378439575-d8705ad7ae80", "0.3,0.8,0.5,0.3,0.4"),
        ("Polarized Aviator Sunglasses", "Apparel", 120.00, "https://images.unsplash.com/photo-1572635196237-14b3f281503f", "0.6,0.7,0.5,0.2,0.7"),
        ("Formal Silk Necktie", "Apparel", 35.00, "https://images.unsplash.com/photo-1589756805437-16474174360e", "0.1,0.95,0.3,0.6,0.3"),
        ("Athletic Compression Shorts", "Apparel", 29.99, "https://images.unsplash.com/photo-1506152983158-b4a74a01c721", "0.5,0.3,0.9,0.1,0.4"),
        ("Graphic Cotton T-Shirt", "Apparel", 24.99, "https://images.unsplash.com/photo-1521572267360-ee0c2909d518", "0.4,0.6,0.5,0.3,0.3"),

        # --- Fitness (10 items) ---
        ("Stainless Steel Water Bottle", "Fitness", 29.99, "https://images.unsplash.com/photo-1602143407151-7111542de6e8", "0.3,0.7,0.8,0.2,0.4"),
        ("Adjustable Dumbbell Set", "Fitness", 179.99, "https://images.unsplash.com/photo-1584735935682-2f2b69dff9d2", "0.6,0.4,0.9,0.1,0.6"),
        ("Yoga Exercise Mat with Strap", "Fitness", 39.99, "https://images.unsplash.com/photo-1601925260368-ae2f83cf8b7f", "0.2,0.6,0.95,0.1,0.4"),
        ("Heavy-Duty Resistance Bands Set", "Fitness", 24.99, "https://images.unsplash.com/photo-1598289431512-b97b09177c45", "0.4,0.3,0.9,0.1,0.5"),
        ("Professional Speed Jump Rope", "Fitness", 19.99, "https://images.unsplash.com/photo-1534438327276-14e5300c3a48", "0.5,0.2,0.95,0.1,0.4"),
        ("Foam Roller for Muscle Recovery", "Fitness", 32.50, "https://images.unsplash.com/photo-1518611012118-696072aa579a", "0.3,0.5,0.85,0.2,0.4"),
        ("Kettlebell 15kg Cast Iron", "Fitness", 69.99, "https://images.unsplash.com/photo-1566241142559-40e1dab266c6", "0.5,0.3,0.9,0.1,0.5"),
        ("Pull-Up Bar Doorway Mount", "Fitness", 45.00, "https://images.unsplash.com/photo-1583454110551-21f2fa2afe61", "0.4,0.4,0.85,0.1,0.5"),
        ("Gym Duffle Bag Waterproof", "Fitness", 55.00, "https://images.unsplash.com/photo-1553062407-98eeb64c6a62", "0.4,0.7,0.8,0.3,0.5"),
        ("Weighted Vest for Running", "Fitness", 89.99, "https://images.unsplash.com/photo-1517838277536-f5f99be501cd", "0.5,0.3,0.95,0.1,0.6")
    ]

    cursor.executemany("""
        INSERT INTO products (name, category, price, image_url, features)
        VALUES (?, ?, ?, ?, ?)
    """, sample_products)

    conn.commit()
    conn.close()
    print("Seeded 30 products (10+ per category across Electronics, Apparel, Fitness) successfully!")

if __name__ == "__main__":
    seed()

Overwriting seed_data.py


In [ ]:
%%writefile app.py
from flask import Flask, render_template_string, request, redirect, url_for, session
import sqlite3
import time
from recommendation_engine import RecommendationEngine, DB_NAME
from seed_data import seed

seed()

app = Flask(__name__)
app.secret_key = "super_secret_ecom_key"

AUTH_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Customer Login / Register</title>
<style>
    body { font-family: 'Segoe UI', sans-serif; background: #f4f7f6; display: flex; justify-content: center; align-items: center; height: 100vh; margin: 0; }
    .auth-box { background: white; padding: 30px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); width: 350px; text-align: center; }
    input { width: 100%; padding: 10px; margin: 10px 0; border: 1px solid #ccc; border-radius: 4px; box-sizing: border-box; }
    button { background: #3498db; color: white; border: none; padding: 10px; width: 100%; border-radius: 4px; font-weight: bold; cursor: pointer; }
    button:hover { background: #2980b9; }
    .error { color: #e74c3c; font-size: 0.9em; margin-bottom: 10px; }
</style>
</head>
<body>
    <div class="auth-box">
        <h2>🛍️ SmartShop AI</h2>
        {% if error %}<div class="error">{{ error }}</div>{% endif %}
        <h3>Login</h3>
        <form action="/login" method="POST">
            <input type="text" name="username" required placeholder="Username">
            <input type="password" name="password" required placeholder="Password">
            <button type="submit">Login</button>
        </form>
        <h3 style="margin-top: 25px;">Register</h3>
        <form action="/register" method="POST">
            <input type="text" name="username" required placeholder="Choose Username">
            <input type="password" name="password" required placeholder="Choose Password">
            <button type="submit" style="background: #2ecc71;">Register & Login</button>
        </form>
    </div>
</body>
</html>
"""

ADMIN_LOGIN_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Admin Login</title>
<style>
    body { font-family: 'Segoe UI', sans-serif; background: #0f172a; display: flex; justify-content: center; align-items: center; height: 100vh; margin: 0; color: white; }
    .auth-box { background: #1e293b; padding: 30px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.3); width: 350px; text-align: center; }
    input { width: 100%; padding: 10px; margin: 10px 0; border: 1px solid #475569; background: #0f172a; color: white; border-radius: 4px; box-sizing: border-box; }
    button { background: #2563eb; color: white; border: none; padding: 10px; width: 100%; border-radius: 4px; font-weight: bold; cursor: pointer; }
</style>
</head>
<body>
    <div class="auth-box">
        <h2>🔒 Admin Portal</h2>
        <p style="font-size: 0.85em; color: #94a3b8;">Default: admin / admin123</p>
        {% if error %}<div style="color: #ef4444; margin-bottom: 10px;">{{ error }}</div>{% endif %}
        <form method="POST">
            <input type="text" name="username" required placeholder="Admin Username">
            <input type="password" name="password" required placeholder="Admin Password">
            <button type="submit">Login</button>
        </form>
    </div>
</body>
</html>
"""

CUSTOMER_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>AI E-Commerce Store</title>
<style>
    body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; background: #f4f7f6; color: #333; }
    header { background: #2c3e50; color: white; padding: 20px; display: flex; justify-content: space-between; align-items: center; }
    .container { max-width: 1200px; margin: 30px auto; padding: 0 20px; }
    h2 { border-bottom: 2px solid #3498db; padding-bottom: 10px; }
    .grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 20px; }
    .card { background: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); overflow: hidden; padding: 15px; text-align: center; }
    .card img { width: 100%; height: 160px; object-fit: cover; border-radius: 4px; }
    .price { color: #e74c3c; font-weight: bold; font-size: 1.1em; margin: 8px 0; }
    .badge { background: #e0f2fe; color: #0369a1; padding: 4px 8px; border-radius: 4px; font-size: 0.8em; }
    .btn-group { display: flex; gap: 4px; margin-top: 10px; }
    button { background: #3498db; color: white; border: none; padding: 8px 6px; border-radius: 4px; cursor: pointer; font-weight: bold; flex: 1; font-size: 0.85em; }
    button:hover { background: #2980b9; }
    .btn-wish { background: #e67e22; }
    .btn-wish:hover { background: #d35400; }
    .btn-buy { background: #27ae60; }
    .btn-buy:hover { background: #219653; }
    .nav-links a { color: #f1c40f; text-decoration: none; margin-left: 15px; font-weight: bold; }
    .count-badge { background: #e74c3c; color: white; border-radius: 50%; padding: 2px 6px; font-size: 0.75em; }
</style>
</head>
<body>
    <header>
        <h1>🛍️ SmartShop AI ({{ username }})</h1>
        <div class="nav-links">
            Wallet: <strong>${{ "%.2f"|format(wallet) }}</strong>
            <a href="/cart">🛒 Cart <span class="count-badge">{{ cart_count }}</span></a>
            <a href="/wishlist">❤️ Wishlist <span class="count-badge">{{ wishlist_count }}</span></a>
            <a href="/logout" style="color: #e74c3c;">Logout</a>
        </div>
    </header>
    <div class="container">
        <h2>🔥 Top 4 Live AI Recommendations</h2>
        <div class="grid">
            {% for p in recommendations %}
            <div class="card">
                <img src="{{ p.image_url }}" alt="{{ p.name }}">
                <h3>{{ p.name }}</h3>
                <span class="badge">{{ p.category }}</span>
                <div class="price">${{ "%.2f"|format(p.price) }}</div>
                <p style="font-size: 0.8em; color: #7f8c8d;">Match Score: {{ p.score }}</p>
                <div class="btn-group">
                    <form action="/interact/{{ p.id }}/cart" method="POST" style="flex:1;"><button type="submit">Cart</button></form>
                    <form action="/interact/{{ p.id }}/wishlist" method="POST" style="flex:1;"><button type="submit" class="btn-wish">Wishlist</button></form>
                    <form action="/interact/{{ p.id }}/purchase" method="POST" style="flex:1;"><button type="submit" class="btn-buy">Buy</button></form>
                </div>
            </div>
            {% endfor %}
        </div>

        <h2 style="margin-top: 50px;">📦 Complete Product Catalog</h2>
        <div class="grid">
            {% for p in all_products %}
            <div class="card">
                <img src="{{ p[4] }}" alt="{{ p[1] }}">
                <h3>{{ p[1] }}</h3>
                <span class="badge">{{ p[2] }}</span>
                <div class="price">${{ "%.2f"|format(p[3]) }}</div>
                <div class="btn-group">
                    <form action="/interact/{{ p[0] }}/cart" method="POST" style="flex:1;"><button type="submit">Cart</button></form>
                    <form action="/interact/{{ p[0] }}/wishlist" method="POST" style="flex:1;"><button type="submit" class="btn-wish">Wishlist</button></form>
                    <form action="/interact/{{ p[0] }}/purchase" method="POST" style="flex:1;"><button type="submit" class="btn-buy">Buy</button></form>
                </div>
            </div>
            {% endfor %}
        </div>
    </div>
</body>
</html>
"""

CART_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Shopping Cart</title>
<style>
    body { font-family: 'Segoe UI', sans-serif; margin: 0; background: #f4f7f6; color: #333; }
    header { background: #2c3e50; color: white; padding: 20px; display: flex; justify-content: space-between; align-items: center; }
    .container { max-width: 900px; margin: 30px auto; padding: 30px; background: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
    table { width: 100%; border-collapse: collapse; margin-bottom: 20px; }
    th, td { padding: 12px; text-align: left; border-bottom: 1px solid #e2e8f0; }
    .total { font-size: 1.4em; font-weight: bold; text-align: right; margin-bottom: 20px; color: #e74c3c; }
    button { background: #27ae60; color: white; border: none; padding: 12px 25px; border-radius: 4px; cursor: pointer; font-weight: bold; font-size: 1.1em; }
    .btn-rem { background: #e74c3c; padding: 6px 12px; font-size: 0.9em; }
</style>
</head>
<body>
    <header><h1>🛒 Shopping Cart</h1><a href="/" style="color: #38bdf8; text-decoration: none; font-weight: bold;">← Back to Store</a></header>
    <div class="container">
        {% if cart_items %}
        <table>
            <tr><th>Product</th><th>Category</th><th>Price</th><th>Action</th></tr>
            {% for item in cart_items %}
            <tr><td>{{ item[1] }}</td><td>{{ item[2] }}</td><td>${{ "%.2f"|format(item[3]) }}</td>
            <td><form action="/cart/remove/{{ item[0] }}" method="POST"><button type="submit" class="btn-rem">Remove</button></form></td></tr>
            {% endfor %}
        </table>
        <div class="total">Total: ${{ "%.2f"|format(total_price) }}</div>
        <form action="/cart/checkout" method="POST" style="text-align: right;"><button type="submit">Proceed to Checkout</button></form>
        {% else %}<p style="text-align: center; font-size: 1.2em; color: #7f8c8d;">Your cart is empty.</p>{% endif %}
    </div>
</body>
</html>
"""

WISHLIST_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>My Wishlist</title>
<style>
    body { font-family: 'Segoe UI', sans-serif; margin: 0; background: #f4f7f6; color: #333; }
    header { background: #2c3e50; color: white; padding: 20px; display: flex; justify-content: space-between; align-items: center; }
    .container { max-width: 1000px; margin: 30px auto; padding: 0 20px; }
    .grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 20px; }
    .card { background: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); overflow: hidden; padding: 15px; text-align: center; }
    .card img { width: 100%; height: 160px; object-fit: cover; border-radius: 4px; }
    .price { color: #e74c3c; font-weight: bold; font-size: 1.1em; margin: 10px 0; }
    .btn-group { display: flex; gap: 8px; margin-top: 10px; }
    button { background: #e74c3c; color: white; border: none; padding: 8px; border-radius: 4px; cursor: pointer; font-weight: bold; flex: 1; font-size: 0.9em; }
    .btn-cart { background: #3498db; }
</style>
</head>
<body>
    <header><h1>❤️ My Wishlist</h1><a href="/" style="color: #38bdf8; text-decoration: none; font-weight: bold;">← Back to Store</a></header>
    <div class="container">
        {% if wishlist_items %}
        <div class="grid">
            {% for p in wishlist_items %}
            <div class="card">
                <img src="{{ p[4] }}" alt="{{ p[1] }}">
                <h3>{{ p[1] }}</h3>
                <div class="price">${{ "%.2f"|format(p[3]) }}</div>
                <div class="btn-group">
                    <form action="/wishlist/move-to-cart/{{ p[0] }}" method="POST" style="flex:1;"><button type="submit" class="btn-cart">Move to Cart</button></form>
                    <form action="/wishlist/remove/{{ p[0] }}" method="POST" style="flex:1;"><button type="submit">Remove</button></form>
                </div>
            </div>
            {% endfor %}
        </div>
        {% else %}<p style="text-align: center; font-size: 1.2em; color: #7f8c8d; margin-top: 50px;">Your wishlist is empty.</p>{% endif %}
    </div>
</body>
</html>
"""

ADMIN_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Admin Dashboard</title>
<style>
    body { font-family: 'Segoe UI', sans-serif; margin: 0; background: #f8fafc; color: #1e293b; }
    header { background: #0f172a; color: white; padding: 20px; display: flex; justify-content: space-between; align-items: center; }
    .container { max-width: 1200px; margin: 30px auto; padding: 0 20px; }
    .section { background: white; padding: 25px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); margin-bottom: 30px; }
    input { padding: 10px; margin: 5px 0 15px 0; width: 100%; box-sizing: border-box; border: 1px solid #cbd5e1; border-radius: 4px; }
    button { background: #2563eb; color: white; border: none; padding: 10px 20px; border-radius: 4px; cursor: pointer; font-weight: bold; }
    button:hover { background: #1d4ed8; }
    table { width: 100%; border-collapse: collapse; margin-top: 15px; }
    th, td { padding: 12px; text-align: left; border-bottom: 1px solid #e2e8f0; }
</style>
</head>
<body>
    <header><h1>⚙️ Admin Dashboard & Product Inventory</h1><a href="/admin/logout" style="color: #ef4444; text-decoration: none; font-weight: bold;">Logout</a></header>
    <div class="container">
        <div class="section">
            <h2>📊 Model Performance Metrics</h2>
            <p><strong>Model Accuracy:</strong> {{ metrics.accuracy }}%</p>
            <p><strong>Model Recall:</strong> {{ metrics.recall }}%</p>
            <div style="text-align: center; margin-top: 20px;">
                <h3>Confusion Matrix Heatmap</h3>
                <img src="data:image/png;base64,{{ metrics.confusion_matrix_plot }}" alt="Confusion Matrix" style="max-width: 400px; border-radius: 6px; border: 1px solid #e2e8f0;">
            </div>
        </div>

        <div class="section">
            <h2>📦 Add New Product</h2>
            <form action="/admin/add-product" method="POST">
                <label>Product Name:</label><input type="text" name="name" required placeholder="e.g., Wireless Earbuds">
                <label>Category:</label><input type="text" name="category" required placeholder="e.g., Electronics">
                <label>Price ($):</label><input type="number" step="0.01" name="price" required placeholder="99.99">
                <label>Image URL:</label><input type="text" name="image_url" required placeholder="https://...">
                <label>Latent Features (5 comma-separated values):</label><input type="text" name="features" required placeholder="0.8,0.2,0.7,0.3,0.5">
                <button type="submit">Add Product</button>
            </form>
        </div>

        <div class="section">
            <h2>Current Inventory Catalog</h2>
            <table>
                <tr><th>ID</th><th>Name</th><th>Category</th><th>Price</th><th>Action</th></tr>
                {% for p in products %}
                <tr>
                    <td>{{ p[0] }}</td><td>{{ p[1] }}</td><td>{{ p[2] }}</td><td>${{ "%.2f"|format(p[3]) }}</td>
                    <td><a href="/admin/delete/{{ p[0] }}" style="color: #ef4444; text-decoration: none; font-weight: bold;">Delete</a></td>
                </tr>
                {% endfor %}
            </table>
        </div>
    </div>
</body>
</html>
"""

@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form['username']
        password = request.form['password']
        conn = sqlite3.connect(DB_NAME, timeout=10.0)
        cursor = conn.cursor()
        cursor.execute("SELECT id FROM users WHERE username = ? AND password = ?", (username, password))
        user = cursor.fetchone()
        conn.close()
        if user:
            session['user_id'] = user[0]
            session['username'] = username
            return redirect(url_for('customer_home'))
        else:
            return render_template_string(AUTH_TEMPLATE, error="Invalid credentials")
    return render_template_string(AUTH_TEMPLATE)

@app.route("/register", methods=["POST"])
def register():
    username = request.form['username']
    password = request.form['password']
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO users (username, password, wallet) VALUES (?, ?, ?)", (username, password, 500.00))
        conn.commit()
        session['user_id'] = cursor.lastrowid
        session['username'] = username
        conn.close()
        return redirect(url_for('customer_home'))
    except sqlite3.IntegrityError:
        conn.close()
        return render_template_string(AUTH_TEMPLATE, error="Username already exists")

@app.route("/logout")
def logout():
    session.clear()
    return redirect(url_for('login'))

@app.route("/")
def customer_home():
    if 'user_id' not in session:
        return redirect(url_for('login'))

    user_id = session['user_id']
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("SELECT id, wallet FROM users WHERE id = ?", (user_id,))
    u_row = cursor.fetchone()
    if not u_row:
        session.clear()
        conn.close()
        return redirect(url_for('login'))

    wallet = u_row[1]
    engine = RecommendationEngine(user_id=user_id)
    recommendations = engine.recommend_products(top_n=4)

    cursor.execute("SELECT COUNT(*) FROM interactions WHERE user_id = ? AND interaction_type = 'cart'", (user_id,))
    cart_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM wishlist WHERE user_id = ?", (user_id,))
    wishlist_count = cursor.fetchone()[0]

    cursor.execute("SELECT id, name, category, price, image_url FROM products")
    all_products = cursor.fetchall()

    for p in all_products:
        cursor.execute("INSERT INTO interactions (user_id, product_id, interaction_type, timestamp) VALUES (?, ?, ?, ?)",
                       (user_id, p[0], 'view', time.time()))
    conn.commit()
    conn.close()

    return render_template_string(CUSTOMER_TEMPLATE, username=session.get('username', 'Customer'), wallet=wallet,
                                  cart_count=cart_count, wishlist_count=wishlist_count,
                                  recommendations=recommendations, all_products=all_products)

@app.route("/cart")
def view_cart():
    if 'user_id' not in session:
        return redirect(url_for('login'))
    user_id = session['user_id']
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("SELECT p.id, p.name, p.category, p.price FROM interactions i JOIN products p ON i.product_id = p.id WHERE i.user_id = ? AND i.interaction_type = 'cart'", (user_id,))
    cart_items = cursor.fetchall()
    total_price = sum([item[3] for item in cart_items])
    conn.close()
    return render_template_string(CART_TEMPLATE, cart_items=cart_items, total_price=total_price)

@app.route("/cart/remove/<int:product_id>", methods=["POST"])
def remove_cart(product_id):
    if 'user_id' not in session:
        return redirect(url_for('login'))
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM interactions WHERE id IN (SELECT id FROM interactions WHERE user_id = ? AND product_id = ? AND interaction_type = 'cart' LIMIT 1)", (session['user_id'], product_id))
    conn.commit()
    conn.close()
    return redirect(url_for('view_cart'))

@app.route("/cart/checkout", methods=["POST"])
def checkout():
    if 'user_id' not in session:
        return redirect(url_for('login'))
    user_id = session['user_id']
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("SELECT p.id, p.price, p.features FROM interactions i JOIN products p ON i.product_id = p.id WHERE i.user_id = ? AND i.interaction_type = 'cart'", (user_id,))
    cart_items = cursor.fetchall()
    if not cart_items:
        conn.close()
        return redirect(url_for('view_cart'))
    total_price = sum([item[1] for item in cart_items])
    cursor.execute("SELECT wallet FROM users WHERE id = ?", (user_id,))
    wallet = cursor.fetchone()[0]
    if wallet >= total_price:
        cursor.execute("UPDATE users SET wallet = wallet - ? WHERE id = ?", (total_price, user_id))
        engine = RecommendationEngine(user_id=user_id)
        for item in cart_items:
            cursor.execute("INSERT INTO interactions (user_id, product_id, interaction_type, timestamp) VALUES (?, ?, 'purchase', ?)", (user_id, item[0], time.time()))
            engine.update_preference_vector(item[2], interaction_weight=1.5)
        cursor.execute("DELETE FROM interactions WHERE user_id = ? AND interaction_type = 'cart'", (user_id,))
        conn.commit()
    conn.close()
    return redirect(url_for('customer_home'))

@app.route("/wishlist")
def view_wishlist():
    if 'user_id' not in session:
        return redirect(url_for('login'))
    user_id = session['user_id']
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("SELECT p.id, p.name, p.category, p.price, p.image_url FROM wishlist w JOIN products p ON w.product_id = p.id WHERE w.user_id = ?", (user_id,))
    wishlist_items = cursor.fetchall()
    conn.close()
    return render_template_string(WISHLIST_TEMPLATE, wishlist_items=wishlist_items)

@app.route("/wishlist/remove/<int:product_id>", methods=["POST"])
def remove_wishlist(product_id):
    if 'user_id' not in session:
        return redirect(url_for('login'))
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM wishlist WHERE user_id = ? AND product_id = ?", (session['user_id'], product_id))
    conn.commit()
    conn.close()
    return redirect(url_for('view_wishlist'))

@app.route("/wishlist/move-to-cart/<int:product_id>", methods=["POST"])
def move_wishlist_to_cart(product_id):
    if 'user_id' not in session:
        return redirect(url_for('login'))
    user_id = session['user_id']
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO interactions (user_id, product_id, interaction_type, timestamp) VALUES (?, ?, 'cart', ?)", (user_id, product_id, time.time()))
    cursor.execute("DELETE FROM wishlist WHERE user_id = ? AND product_id = ?", (user_id, product_id))
    conn.commit()
    conn.close()
    return redirect(url_for('view_wishlist'))

@app.route("/interact/<int:product_id>/<action>", methods=["POST"])
def interact(product_id, action):
    if 'user_id' not in session:
        return redirect(url_for('login'))
    user_id = session['user_id']
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("SELECT features, price FROM products WHERE id = ?", (product_id,))
    row = cursor.fetchone()
    if row:
        features, price = row[0], row[1]
        if action == 'wishlist':
            try: cursor.execute("INSERT INTO wishlist (user_id, product_id) VALUES (?, ?)", (user_id, product_id))
            except sqlite3.IntegrityError: pass
            i_type, weight = 'wishlist', 0.8
        elif action == 'cart':
            i_type, weight = 'cart', 1.0
        elif action == 'purchase':
            i_type, weight = 'purchase', 1.5
            cursor.execute("UPDATE users SET wallet = wallet - ? WHERE id = ? AND wallet >= ?", (price, user_id, price))
        else:
            i_type, weight = 'view', 0.2
        cursor.execute("INSERT INTO interactions (user_id, product_id, interaction_type, timestamp) VALUES (?, ?, ?, ?)", (user_id, product_id, i_type, time.time()))
        conn.commit()
        engine = RecommendationEngine(user_id=user_id)
        engine.update_preference_vector(features, interaction_weight=weight)
    conn.close()
    return redirect(url_for('customer_home'))

@app.route("/admin/login", methods=["GET", "POST"])
def admin_login():
    if request.method == "POST":
        if request.form['username'] == "admin" and request.form['password'] == "admin123":
            session['is_admin'] = True
            return redirect(url_for('admin_portal'))
        else:
            return render_template_string(ADMIN_LOGIN_TEMPLATE, error="Invalid credentials")
    return render_template_string(ADMIN_LOGIN_TEMPLATE)

@app.route("/admin/logout")
def admin_logout():
    session.pop('is_admin', None)
    return redirect(url_for('admin_login'))

@app.route("/admin")
def admin_portal():
    if not session.get('is_admin'):
        return redirect(url_for('admin_login'))
    engine = RecommendationEngine(user_id=1)
    metrics = engine.evaluate_model()

    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("SELECT id, name, category, price, image_url FROM products")
    products = cursor.fetchall()
    conn.close()

    return render_template_string(ADMIN_TEMPLATE, metrics=metrics, products=products)

@app.route("/admin/add-product", methods=["POST"])
def add_product():
    if not session.get('is_admin'):
        return redirect(url_for('admin_login'))
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO products (name, category, price, image_url, features) VALUES (?, ?, ?, ?, ?)",
                   (request.form['name'], request.form['category'], float(request.form['price']), request.form['image_url'], request.form['features']))
    conn.commit()
    conn.close()
    return redirect(url_for('admin_portal'))

@app.route("/admin/delete/<int:product_id>")
def delete_product(product_id):
    if not session.get('is_admin'):
        return redirect(url_for('admin_login'))
    conn = sqlite3.connect(DB_NAME, timeout=10.0)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM products WHERE id = ?", (product_id,))
    conn.commit()
    conn.close()
    return redirect(url_for('admin_portal'))

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

Overwriting app.py


In [ ]:
%%writefile train_model.py
import pandas as pd
import sqlite3
import time
from database import DB_NAME, init_db

def train_model_from_excel():
    init_db()

    try:
        # Load the training dataset from Excel
        train_df = pd.read_excel("Dataset_training.xlsx")
        print(f"Successfully loaded Dataset_training.xlsx with {len(train_df)} rows.")
        print("Dataset Columns Found:", list(train_df.columns))

        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()

        # Check if interactions table already has training data
        cursor.execute("SELECT COUNT(*) FROM interactions")
        count = cursor.fetchone()[0]

        if count == 0 and len(train_df) > 0:
            # Map Excel columns to database interaction logs if columns exist
            # Adjust column mapping below to match your exact Excel sheet headers if necessary
            user_col = 'user_id' if 'user_id' in train_df.columns else train_df.columns[0]
            prod_col = 'product_id' if 'product_id' in train_df.columns else train_df.columns[1]
            type_col = 'interaction_type' if 'interaction_type' in train_df.columns else train_df.columns[-1]

            interactions_to_insert = []
            for _, row in train_df.iterrows():
                u_id = int(row[user_col]) if pd.notna(row[user_col]) else 1
                p_id = int(row[prod_col]) if pd.notna(row[prod_col]) else 1
                i_type = str(row[type_col]) if pd.notna(row[type_col]) else 'cart'
                interactions_to_insert.append((u_id, p_id, i_type, time.time()))

            cursor.executemany("""
                INSERT INTO interactions (user_id, product_id, interaction_type, timestamp)
                VALUES (?, ?, ?, ?)
            """, interactions_to_insert)

            conn.commit()
            print(f"Imported {len(interactions_to_insert)} training interactions into database!")
        else:
            print("Interactions table already populated or training set is empty.")

        conn.close()

    except FileNotFoundError:
        print("⚠️ Warning: 'Dataset_training.xlsx' not found in the current directory. Please upload your Excel file to the environment.")
    except Exception as e:
        print(f"Error processing training dataset: {e}")

if __name__ == "__main__":
    train_model_from_excel()

Overwriting train_model.py


In [ ]:
!pip install -q scikit-learn matplotlib seaborn numpy pandas openpyxl

In [ ]:
import subprocess
import threading
import time
import re
import os

# 1. Clear old ports and cloudflared processes
subprocess.run(["fuser", "-k", "5000/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["fuser", "-k", "5001/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 2. Ensure cloudflared binary exists
if not os.path.exists('/content/cloudflared'):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/content/cloudflared"])
    subprocess.run(["chmod", "+x", "/content/cloudflared"])

# 3. Start Flask apps in background threads
def run_user():
    import app
    app.app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

def run_admin():
    import app
    # Running admin logic/portal on port 5001
    app.app.run(host='127.0.0.1', port=5001, debug=False, use_reloader=False)

threading.Thread(target=run_user, daemon=True).start()
threading.Thread(target=run_admin, daemon=True).start()
time.sleep(2)

# 4. Start Cloudflare Tunnels for both ports
t1 = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:5000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
t2 = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:5001"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def extract_url(proc):
    start = time.time()
    while time.time() - start < 25:
        line = proc.stderr.readline()
        if not line:
            time.sleep(0.1)
            continue
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m: return m.group(0)
    return None

print("⏳ Generating live Cloudflare links...")
url_user = extract_url(t1)
url_admin = extract_url(t2)

print("\n" + "="*75)
if url_user:
    print(f"🛍️ CUSTOMER APP LINK: {url_user}/login  (Login: customer / user123)")
if url_admin:
    print(f"🔒 ADMIN APP LINK:    {url_admin}/admin/login  (Login: admin / admin123)")
print("="*75)

Recommendation Engine updated!
Database & tables initialized successfully with Auth & Wishlist support!
 * Serving Flask app 'app'
 * Debug mode: off
 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:Press CTRL+C to quit


⏳ Generating live Cloudflare links...

🛍️ CUSTOMER APP LINK: https://david-conventions-its-carrier.trycloudflare.com/login  (Login: customer / user123)
🔒 ADMIN APP LINK:    https://airlines-innovative-kissing-edwards.trycloudflare.com/admin/login  (Login: admin / admin123)
